# AIDB Budgets

Making sure AI Agents don't result in a suprise bill because of an excessive number of tokens being used is always a concern.

With AIDB Agents, you can set the maximum input and output tokens as well as the maximum number of iterations. In addition to the Model chosen, these are the three main variables that influence the number of tokens used and thus, the expense.
    
This demo will use the Semantic KB to show how these bugets work.

In this demo, a semantic KB over the `supply_chain` schema powers an AI agent that discovers the relevant tables, writes SQL, and runs it. Because comments are embedded alongside the structure, well-commented tables give better search results.

## What you will do in this notebook

1. Create a realistic **supply chain schema** with a comment on every table and column.
2. Load **sample data** 
3. Create the **semantic KB** over that schema.
4. Create a **Claude model** and an **agent** that uses the KB's search tools together with a SQL tool.
5. **Ask questions** in plain English, including a follow-up that depends on the previous answer.
6. Define budgets for input and output prompts as well as max iterations.

## Prerequisites

- A Postgres database with AIDB installed.
- An Anthropic (Claude) API key, which you will be prompted for in the next cell.

## Claude API Key

Run the next cell and paste your Anthropic API key when prompted. `getpass` hides the input, and the key is kept in memory only. It is wrapped in a JSON document (`creds`) that a later cell passes to `aidb.create_model` as the named parameter `:creds`.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

%config SqlMagic.named_parameters = "enabled"

## Connect to the database

Creates a SQLAlchemy engine for the `aidb` database (connecting as `postgres` or `gpadmin` over the local socket) and attaches the `%sql` / `%%sql` magics to it. Adjust the connection variables if your database is set up differently.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Supply chain schema (`supply_chain`)

| Area | Table | Purpose |
|---|---|---|
| Sourcing | `suppliers` | Vendors we buy from, with status, rating and payment terms |
| | `supplier_products` | Which suppliers offer which products, with price, lead time and minimum order quantity |
| | `product_categories` | Hierarchical product categories |
| | `products` | Product catalog by SKU, with cost, list price and weight |
| Stock | `warehouses` | Warehouses and distribution centers |
| | `inventory` | Current on-hand and reserved stock per warehouse and product, with reorder settings |
| | `inventory_transactions` | Ledger of every stock movement (receipts, shipments, adjustments, transfers, returns) |
| Purchasing | `purchase_orders` | Orders placed with suppliers to restock a warehouse |
| | `purchase_order_lines` | Products, quantities and costs on each purchase order |
| | `purchase_receipts` | Deliveries received against PO lines, including rejected units |
| Sales | `customers` | Customer accounts by segment, with credit limit and status |
| | `sales_orders` | Customer order headers: status, fulfilling warehouse, ship-to address, total |
| | `sales_order_lines` | Products, quantities, prices and discounts on each sales order |
| Delivery | `carriers` | Shipping companies |
| | `shipments` | Outbound packages per sales order, with tracking, status and delivery times |
| | `shipment_lines` | Which order lines and quantities are in each shipment |

**Key relationships**
- `suppliers` → `purchase_orders` → `purchase_order_lines` → `purchase_receipts`
- `customers` → `sales_orders` → `sales_order_lines` → `shipment_lines` ← `shipments` → `carriers`
- `products` links to supplier offers, inventory, and both kinds of order line
- `warehouses` links to inventory, purchase orders (destination), sales orders (fulfillment) and shipments (origin)

**Sample questions it supports**
- Which products are at or below their reorder point?
- Which suppliers deliver late most often? (`purchase_receipts.received_at` vs `purchase_orders.expected_delivery_date`)
- What is the on-time delivery rate by carrier? (`shipments.delivered_at` vs `estimated_delivery_at`)
- What are the top customers by order value?

In [ ]:
%%sql

DROP SCHEMA IF EXISTS supply_chain CASCADE;

CREATE SCHEMA supply_chain;

COMMENT ON SCHEMA supply_chain IS
'Operational (OLTP) supply chain data: suppliers and the products they sell, the product catalog, warehouse inventory and stock movements, purchase orders placed with suppliers to replenish stock, customers and their sales orders, and outbound shipments delivered by carriers. Use this schema to answer questions about stock levels, supplier performance, order fulfillment, and delivery status.';

SET search_path = supply_chain, public;

-- -----------------------------------------------------------------------------
-- suppliers
-- -----------------------------------------------------------------------------
CREATE TABLE suppliers (
    supplier_id         bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    supplier_code       text NOT NULL,
    supplier_name       text NOT NULL,
    country_code        char(2) NOT NULL,
    contact_email       text,
    contact_phone       text,
    payment_terms_days  integer NOT NULL DEFAULT 30 CHECK (payment_terms_days >= 0),
    status              text NOT NULL DEFAULT 'active'
                        CHECK (status IN ('active', 'on_hold', 'inactive')),
    rating              numeric(3,2) CHECK (rating BETWEEN 0 AND 5),
    created_at          timestamptz NOT NULL DEFAULT now(),
    updated_at          timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE suppliers IS
'Vendors that sell products to the company. One row per supplier. Referenced by supplier_products (what they sell) and purchase_orders (what we bought from them).';
COMMENT ON COLUMN suppliers.supplier_id IS 'Surrogate primary key for the supplier.';
COMMENT ON COLUMN suppliers.supplier_code IS 'Short unique business code identifying the supplier, such as SUP-0001.';
COMMENT ON COLUMN suppliers.supplier_name IS 'Legal or trading name of the supplier company.';
COMMENT ON COLUMN suppliers.country_code IS 'Two-letter ISO 3166-1 country code where the supplier is based, such as US, DE, CN.';
COMMENT ON COLUMN suppliers.contact_email IS 'Primary email address for purchasing contact at the supplier.';
COMMENT ON COLUMN suppliers.contact_phone IS 'Primary phone number for purchasing contact at the supplier.';
COMMENT ON COLUMN suppliers.payment_terms_days IS 'Number of days after invoice date that payment to this supplier is due (for example 30 means net-30).';
COMMENT ON COLUMN suppliers.status IS 'Supplier lifecycle status: active (can be ordered from), on_hold (temporarily blocked), inactive (no longer used).';
COMMENT ON COLUMN suppliers.rating IS 'Internal performance score from 0.00 (worst) to 5.00 (best) based on delivery reliability and quality.';
COMMENT ON COLUMN suppliers.created_at IS 'Timestamp when the supplier record was created.';
COMMENT ON COLUMN suppliers.updated_at IS 'Timestamp when the supplier record was last modified.';

-- -----------------------------------------------------------------------------
-- product_categories
-- -----------------------------------------------------------------------------
CREATE TABLE product_categories (
    category_id         bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    category_name       text NOT NULL,
    parent_category_id  bigint REFERENCES product_categories (category_id),
    description         text
);

COMMENT ON TABLE product_categories IS
'Hierarchical product classification (for example Electronics > Cables). A category with a NULL parent_category_id is a top-level category.';
COMMENT ON COLUMN product_categories.category_id IS 'Surrogate primary key for the category.';
COMMENT ON COLUMN product_categories.category_name IS 'Unique human-readable category name.';
COMMENT ON COLUMN product_categories.parent_category_id IS 'Category this one belongs under; references product_categories.category_id. NULL for top-level categories.';
COMMENT ON COLUMN product_categories.description IS 'Free-text description of what products belong in this category.';

-- -----------------------------------------------------------------------------
-- products
-- -----------------------------------------------------------------------------
CREATE TABLE products (
    product_id      bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    sku             text NOT NULL,
    product_name    text NOT NULL,
    description     text,
    category_id     bigint NOT NULL REFERENCES product_categories (category_id),
    unit_of_measure text NOT NULL DEFAULT 'each'
                    CHECK (unit_of_measure IN ('each', 'box', 'pallet', 'kg', 'liter')),
    unit_cost       numeric(12,2) NOT NULL CHECK (unit_cost >= 0),
    list_price      numeric(12,2) NOT NULL CHECK (list_price >= 0),
    weight_kg       numeric(10,3) CHECK (weight_kg >= 0),
    is_active       boolean NOT NULL DEFAULT true,
    created_at      timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE products IS
'Product catalog: every item the company stocks and sells. One row per SKU. Stock quantities live in inventory, not here.';
COMMENT ON COLUMN products.product_id IS 'Surrogate primary key for the product.';
COMMENT ON COLUMN products.sku IS 'Stock keeping unit: unique business identifier printed on labels and used on orders.';
COMMENT ON COLUMN products.product_name IS 'Short display name of the product.';
COMMENT ON COLUMN products.description IS 'Longer free-text description of the product.';
COMMENT ON COLUMN products.category_id IS 'Category the product belongs to; references product_categories.category_id.';
COMMENT ON COLUMN products.unit_of_measure IS 'Unit in which the product is stocked and sold: each, box, pallet, kg, or liter.';
COMMENT ON COLUMN products.unit_cost IS 'Standard internal cost to acquire one unit, in the base currency (USD). Used for margin and inventory valuation.';
COMMENT ON COLUMN products.list_price IS 'Default selling price for one unit before customer discounts, in the base currency (USD).';
COMMENT ON COLUMN products.weight_kg IS 'Weight of one unit in kilograms; used to estimate shipping cost.';
COMMENT ON COLUMN products.is_active IS 'True if the product can currently be sold and ordered; false if discontinued.';
COMMENT ON COLUMN products.created_at IS 'Timestamp when the product was added to the catalog.';

CREATE INDEX products_category_idx ON products (category_id);

-- -----------------------------------------------------------------------------
-- supplier_products
-- -----------------------------------------------------------------------------
CREATE TABLE supplier_products (
    supplier_product_id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    supplier_id         bigint NOT NULL REFERENCES suppliers (supplier_id),
    product_id          bigint NOT NULL REFERENCES products (product_id),
    supplier_sku        text,
    unit_price          numeric(12,2) NOT NULL CHECK (unit_price >= 0),
    currency_code       char(3) NOT NULL DEFAULT 'USD',
    lead_time_days      integer NOT NULL CHECK (lead_time_days >= 0),
    min_order_qty       integer NOT NULL DEFAULT 1 CHECK (min_order_qty >= 1),
    is_preferred        boolean NOT NULL DEFAULT false
);

COMMENT ON TABLE supplier_products IS
'Sourcing catalog linking suppliers to the products they can supply, with the price, lead time, and minimum order quantity each supplier offers. A product may have several suppliers.';
COMMENT ON COLUMN supplier_products.supplier_product_id IS 'Surrogate primary key for the supplier-product offer.';
COMMENT ON COLUMN supplier_products.supplier_id IS 'Supplier making the offer; references suppliers.supplier_id.';
COMMENT ON COLUMN supplier_products.product_id IS 'Product being offered; references products.product_id.';
COMMENT ON COLUMN supplier_products.supplier_sku IS 'The supplier''s own part number for this product, if different from our SKU.';
COMMENT ON COLUMN supplier_products.unit_price IS 'Price the supplier charges per unit, in currency_code.';
COMMENT ON COLUMN supplier_products.currency_code IS 'Three-letter ISO 4217 currency code of unit_price, such as USD or EUR.';
COMMENT ON COLUMN supplier_products.lead_time_days IS 'Typical number of days between placing an order with this supplier and receiving the goods.';
COMMENT ON COLUMN supplier_products.min_order_qty IS 'Smallest quantity the supplier will accept on one order line.';
COMMENT ON COLUMN supplier_products.is_preferred IS 'True if this supplier is the preferred source for the product.';

CREATE INDEX supplier_products_product_idx ON supplier_products (product_id);

-- -----------------------------------------------------------------------------
-- warehouses
-- -----------------------------------------------------------------------------
CREATE TABLE warehouses (
    warehouse_id        bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    warehouse_code      text NOT NULL,
    warehouse_name      text NOT NULL,
    city                text NOT NULL,
    state_province      text,
    country_code        char(2) NOT NULL,
    capacity_units      integer CHECK (capacity_units > 0),
    is_active           boolean NOT NULL DEFAULT true
);

COMMENT ON TABLE warehouses IS
'Physical warehouses or distribution centers that store inventory, receive purchase orders, and ship customer orders.';
COMMENT ON COLUMN warehouses.warehouse_id IS 'Surrogate primary key for the warehouse.';
COMMENT ON COLUMN warehouses.warehouse_code IS 'Short unique business code for the warehouse, such as WH-DAL.';
COMMENT ON COLUMN warehouses.warehouse_name IS 'Display name of the warehouse.';
COMMENT ON COLUMN warehouses.city IS 'City where the warehouse is located.';
COMMENT ON COLUMN warehouses.state_province IS 'State or province where the warehouse is located.';
COMMENT ON COLUMN warehouses.country_code IS 'Two-letter ISO 3166-1 country code of the warehouse location.';
COMMENT ON COLUMN warehouses.capacity_units IS 'Maximum number of product units the warehouse can hold.';
COMMENT ON COLUMN warehouses.is_active IS 'True if the warehouse is operating; false if closed.';

-- -----------------------------------------------------------------------------
-- inventory
-- -----------------------------------------------------------------------------
CREATE TABLE inventory (
    warehouse_id        bigint NOT NULL REFERENCES warehouses (warehouse_id),
    product_id          bigint NOT NULL REFERENCES products (product_id),
    quantity_on_hand    integer NOT NULL DEFAULT 0 CHECK (quantity_on_hand >= 0),
    quantity_reserved   integer NOT NULL DEFAULT 0 CHECK (quantity_reserved >= 0),
    reorder_point       integer NOT NULL DEFAULT 0 CHECK (reorder_point >= 0),
    reorder_quantity    integer NOT NULL DEFAULT 0 CHECK (reorder_quantity >= 0),
    last_counted_at     timestamptz,
    updated_at          timestamptz NOT NULL DEFAULT now(),
    PRIMARY KEY (warehouse_id, product_id),
    CHECK (quantity_reserved <= quantity_on_hand)
);

COMMENT ON TABLE inventory IS
'Current stock level of each product in each warehouse. One row per warehouse and product pair. Available-to-sell stock is quantity_on_hand minus quantity_reserved. Changes are explained by rows in inventory_transactions.';
COMMENT ON COLUMN inventory.warehouse_id IS 'Warehouse holding the stock; references warehouses.warehouse_id. Part of the primary key.';
COMMENT ON COLUMN inventory.product_id IS 'Product being stocked; references products.product_id. Part of the primary key.';
COMMENT ON COLUMN inventory.quantity_on_hand IS 'Units physically present in the warehouse right now.';
COMMENT ON COLUMN inventory.quantity_reserved IS 'Units on hand that are allocated to confirmed sales orders but not yet shipped. Never exceeds quantity_on_hand.';
COMMENT ON COLUMN inventory.reorder_point IS 'When available stock falls to or below this number, the product should be reordered from a supplier.';
COMMENT ON COLUMN inventory.reorder_quantity IS 'Suggested quantity to order when the reorder point is reached.';
COMMENT ON COLUMN inventory.last_counted_at IS 'Timestamp of the most recent physical cycle count for this product in this warehouse.';
COMMENT ON COLUMN inventory.updated_at IS 'Timestamp when this stock row was last changed.';

CREATE INDEX inventory_product_idx ON inventory (product_id);

-- -----------------------------------------------------------------------------
-- inventory_transactions
-- -----------------------------------------------------------------------------
CREATE TABLE inventory_transactions (
    transaction_id      bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    warehouse_id        bigint NOT NULL REFERENCES warehouses (warehouse_id),
    product_id          bigint NOT NULL REFERENCES products (product_id),
    transaction_type    text NOT NULL
                        CHECK (transaction_type IN ('receipt', 'shipment', 'adjustment', 'transfer_in', 'transfer_out', 'return')),
    quantity_change     integer NOT NULL CHECK (quantity_change <> 0),
    reference_type      text CHECK (reference_type IN ('purchase_order', 'sales_order', 'manual')),
    reference_id        bigint,
    notes               text,
    created_at          timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE inventory_transactions IS
'Append-only ledger of every stock movement: goods received from suppliers, goods shipped to customers, adjustments, transfers between warehouses, and customer returns. Summing quantity_change per warehouse and product reproduces inventory.quantity_on_hand.';
COMMENT ON COLUMN inventory_transactions.transaction_id IS 'Surrogate primary key for the stock movement.';
COMMENT ON COLUMN inventory_transactions.warehouse_id IS 'Warehouse where the stock moved; references warehouses.warehouse_id.';
COMMENT ON COLUMN inventory_transactions.product_id IS 'Product that moved; references products.product_id.';
COMMENT ON COLUMN inventory_transactions.transaction_type IS 'Kind of movement: receipt (supplier delivery), shipment (customer order leaves), adjustment (count correction or damage), transfer_in or transfer_out (between warehouses), return (customer return).';
COMMENT ON COLUMN inventory_transactions.quantity_change IS 'Signed number of units: positive adds stock, negative removes stock. Never zero.';
COMMENT ON COLUMN inventory_transactions.reference_type IS 'Kind of business document that caused the movement: purchase_order, sales_order, or manual (no document).';
COMMENT ON COLUMN inventory_transactions.reference_id IS 'Id of the purchase_orders or sales_orders row named by reference_type. NULL for manual movements. Not a declared foreign key because it points to different tables.';
COMMENT ON COLUMN inventory_transactions.notes IS 'Free-text explanation, mainly for manual adjustments.';
COMMENT ON COLUMN inventory_transactions.created_at IS 'Timestamp when the movement was recorded.';

CREATE INDEX inventory_transactions_wh_prod_idx ON inventory_transactions (warehouse_id, product_id, created_at);

-- -----------------------------------------------------------------------------
-- customers
-- -----------------------------------------------------------------------------
CREATE TABLE customers (
    customer_id             bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_code           text NOT NULL,
    customer_name           text NOT NULL,
    customer_type           text NOT NULL DEFAULT 'retail'
                            CHECK (customer_type IN ('retail', 'wholesale', 'distributor')),
    email                   text,
    phone                   text,
    billing_city            text,
    billing_state_province  text,
    billing_country_code    char(2),
    credit_limit            numeric(14,2) NOT NULL DEFAULT 0 CHECK (credit_limit >= 0),
    status                  text NOT NULL DEFAULT 'active'
                            CHECK (status IN ('active', 'suspended', 'closed')),
    created_at              timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE customers IS
'Companies or individuals that buy products. One row per customer account. Referenced by sales_orders.';
COMMENT ON COLUMN customers.customer_id IS 'Surrogate primary key for the customer.';
COMMENT ON COLUMN customers.customer_code IS 'Short unique business code for the customer account, such as CUST-00421.';
COMMENT ON COLUMN customers.customer_name IS 'Name of the customer company or person.';
COMMENT ON COLUMN customers.customer_type IS 'Customer segment: retail (individual or small buyer), wholesale (bulk buyer), or distributor (resells to others).';
COMMENT ON COLUMN customers.email IS 'Primary contact email address for the customer.';
COMMENT ON COLUMN customers.phone IS 'Primary contact phone number for the customer.';
COMMENT ON COLUMN customers.billing_city IS 'City of the customer billing address.';
COMMENT ON COLUMN customers.billing_state_province IS 'State or province of the customer billing address.';
COMMENT ON COLUMN customers.billing_country_code IS 'Two-letter ISO 3166-1 country code of the billing address.';
COMMENT ON COLUMN customers.credit_limit IS 'Maximum unpaid order value the customer may carry, in USD. Zero means prepay only.';
COMMENT ON COLUMN customers.status IS 'Account status: active (can order), suspended (blocked, for example overdue payment), closed (no longer a customer).';
COMMENT ON COLUMN customers.created_at IS 'Timestamp when the customer account was created.';

-- -----------------------------------------------------------------------------
-- sales_orders
-- -----------------------------------------------------------------------------
CREATE TABLE sales_orders (
    sales_order_id          bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    order_number            text NOT NULL,
    customer_id             bigint NOT NULL REFERENCES customers (customer_id),
    warehouse_id            bigint NOT NULL REFERENCES warehouses (warehouse_id),
    order_status            text NOT NULL DEFAULT 'pending'
                            CHECK (order_status IN ('pending', 'confirmed', 'picking', 'shipped', 'delivered', 'cancelled')),
    order_date              timestamptz NOT NULL DEFAULT now(),
    requested_delivery_date date,
    ship_to_city            text,
    ship_to_state_province  text,
    ship_to_country_code    char(2),
    total_amount            numeric(14,2) NOT NULL DEFAULT 0 CHECK (total_amount >= 0),
    currency_code           char(3) NOT NULL DEFAULT 'USD',
    created_at              timestamptz NOT NULL DEFAULT now(),
    updated_at              timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE sales_orders IS
'Header of a customer order: who ordered, which warehouse fulfills it, its status, and where it ships. Line items are in sales_order_lines; deliveries are in shipments.';
COMMENT ON COLUMN sales_orders.sales_order_id IS 'Surrogate primary key for the sales order.';
COMMENT ON COLUMN sales_orders.order_number IS 'Unique human-readable order number shown to the customer, such as SO-2026-000123.';
COMMENT ON COLUMN sales_orders.customer_id IS 'Customer who placed the order; references customers.customer_id.';
COMMENT ON COLUMN sales_orders.warehouse_id IS 'Warehouse that fulfills and ships the order; references warehouses.warehouse_id.';
COMMENT ON COLUMN sales_orders.order_status IS 'Fulfillment status: pending (received, not yet approved), confirmed (stock reserved), picking (being prepared), shipped (left the warehouse), delivered (received by customer), cancelled.';
COMMENT ON COLUMN sales_orders.order_date IS 'Timestamp when the customer placed the order.';
COMMENT ON COLUMN sales_orders.requested_delivery_date IS 'Date by which the customer asked to receive the goods.';
COMMENT ON COLUMN sales_orders.ship_to_city IS 'City of the delivery address.';
COMMENT ON COLUMN sales_orders.ship_to_state_province IS 'State or province of the delivery address.';
COMMENT ON COLUMN sales_orders.ship_to_country_code IS 'Two-letter ISO 3166-1 country code of the delivery address.';
COMMENT ON COLUMN sales_orders.total_amount IS 'Total order value after discounts, equal to the sum of line_total across its sales_order_lines, in currency_code.';
COMMENT ON COLUMN sales_orders.currency_code IS 'Three-letter ISO 4217 currency code for all amounts on the order.';
COMMENT ON COLUMN sales_orders.created_at IS 'Timestamp when the order record was created.';
COMMENT ON COLUMN sales_orders.updated_at IS 'Timestamp when the order record was last modified, for example on a status change.';

CREATE INDEX sales_orders_customer_idx ON sales_orders (customer_id);
CREATE INDEX sales_orders_warehouse_idx ON sales_orders (warehouse_id);
CREATE INDEX sales_orders_status_date_idx ON sales_orders (order_status, order_date);

-- -----------------------------------------------------------------------------
-- sales_order_lines
-- -----------------------------------------------------------------------------
CREATE TABLE sales_order_lines (
    sales_order_line_id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    sales_order_id      bigint NOT NULL REFERENCES sales_orders (sales_order_id) ON DELETE CASCADE,
    line_number         integer NOT NULL CHECK (line_number > 0),
    product_id          bigint NOT NULL REFERENCES products (product_id),
    quantity_ordered    integer NOT NULL CHECK (quantity_ordered > 0),
    unit_price          numeric(12,2) NOT NULL CHECK (unit_price >= 0),
    discount_pct        numeric(5,2) NOT NULL DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    line_total          numeric(14,2) GENERATED ALWAYS AS
                        (round(quantity_ordered * unit_price * (1 - discount_pct / 100), 2)) STORED
);

COMMENT ON TABLE sales_order_lines IS
'Line items of a sales order: one row per product on the order, with quantity, price, and discount.';
COMMENT ON COLUMN sales_order_lines.sales_order_line_id IS 'Surrogate primary key for the order line.';
COMMENT ON COLUMN sales_order_lines.sales_order_id IS 'Order this line belongs to; references sales_orders.sales_order_id.';
COMMENT ON COLUMN sales_order_lines.line_number IS 'Position of the line within its order, starting at 1.';
COMMENT ON COLUMN sales_order_lines.product_id IS 'Product being sold; references products.product_id.';
COMMENT ON COLUMN sales_order_lines.quantity_ordered IS 'Number of units the customer ordered.';
COMMENT ON COLUMN sales_order_lines.unit_price IS 'Price per unit charged to the customer before discount, in the order currency.';
COMMENT ON COLUMN sales_order_lines.discount_pct IS 'Percentage discount applied to this line, from 0 to 100.';
COMMENT ON COLUMN sales_order_lines.line_total IS 'Computed line amount: quantity_ordered times unit_price less the discount, rounded to 2 decimals.';

CREATE INDEX sales_order_lines_product_idx ON sales_order_lines (product_id);

-- -----------------------------------------------------------------------------
-- purchase_orders
-- -----------------------------------------------------------------------------
CREATE TABLE purchase_orders (
    purchase_order_id       bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    po_number               text NOT NULL,
    supplier_id             bigint NOT NULL REFERENCES suppliers (supplier_id),
    warehouse_id            bigint NOT NULL REFERENCES warehouses (warehouse_id),
    po_status               text NOT NULL DEFAULT 'draft'
                            CHECK (po_status IN ('draft', 'submitted', 'confirmed', 'partially_received', 'received', 'cancelled')),
    order_date              date NOT NULL DEFAULT current_date,
    expected_delivery_date  date,
    total_amount            numeric(14,2) NOT NULL DEFAULT 0 CHECK (total_amount >= 0),
    currency_code           char(3) NOT NULL DEFAULT 'USD',
    created_at              timestamptz NOT NULL DEFAULT now(),
    updated_at              timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE purchase_orders IS
'This is the purchase orders table. Header of an order the company places with a supplier to replenish stock at a warehouse. Line items are in purchase_order_lines; deliveries received against it are in purchase_receipts.';
COMMENT ON COLUMN purchase_orders.purchase_order_id IS 'Surrogate primary key for the purchase order.';
COMMENT ON COLUMN purchase_orders.po_number IS 'Unique human-readable purchase order number sent to the supplier, such as PO-2026-000045.';
COMMENT ON COLUMN purchase_orders.supplier_id IS 'Supplier the goods are ordered from; references suppliers.supplier_id.';
COMMENT ON COLUMN purchase_orders.warehouse_id IS 'Destination warehouse where the goods will be delivered; references warehouses.warehouse_id.';
COMMENT ON COLUMN purchase_orders.po_status IS 'Purchasing status: draft (being prepared), submitted (sent to supplier), confirmed (supplier accepted), partially_received (some lines delivered), received (all delivered), cancelled.';
COMMENT ON COLUMN purchase_orders.order_date IS 'Date the purchase order was issued to the supplier.';
COMMENT ON COLUMN purchase_orders.expected_delivery_date IS 'Date the supplier promised to deliver. Compare with purchase_receipts.received_at to measure supplier lateness.';
COMMENT ON COLUMN purchase_orders.total_amount IS 'Total purchase value, equal to the sum of line_total across its purchase_order_lines, in currency_code.';
COMMENT ON COLUMN purchase_orders.currency_code IS 'Three-letter ISO 4217 currency code for all amounts on the purchase order.';
COMMENT ON COLUMN purchase_orders.created_at IS 'Timestamp when the purchase order record was created.';
COMMENT ON COLUMN purchase_orders.updated_at IS 'Timestamp when the purchase order record was last modified.';

CREATE INDEX purchase_orders_supplier_idx ON purchase_orders (supplier_id);
CREATE INDEX purchase_orders_warehouse_idx ON purchase_orders (warehouse_id);
CREATE INDEX purchase_orders_status_idx ON purchase_orders (po_status, expected_delivery_date);

-- -----------------------------------------------------------------------------
-- purchase_order_lines
-- -----------------------------------------------------------------------------
CREATE TABLE purchase_order_lines (
    po_line_id          bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    purchase_order_id   bigint NOT NULL REFERENCES purchase_orders (purchase_order_id) ON DELETE CASCADE,
    line_number         integer NOT NULL CHECK (line_number > 0),
    product_id          bigint NOT NULL REFERENCES products (product_id),
    quantity_ordered    integer NOT NULL CHECK (quantity_ordered > 0),
    quantity_received   integer NOT NULL DEFAULT 0 CHECK (quantity_received >= 0),
    unit_cost           numeric(12,2) NOT NULL CHECK (unit_cost >= 0),
    line_total          numeric(14,2) GENERATED ALWAYS AS (round(quantity_ordered * unit_cost, 2)) STORED
);

COMMENT ON TABLE purchase_order_lines IS
'Line items of a purchase order: one row per product ordered from the supplier, with quantity ordered, quantity received so far, and agreed unit cost.';
COMMENT ON COLUMN purchase_order_lines.po_line_id IS 'Surrogate primary key for the purchase order line.';
COMMENT ON COLUMN purchase_order_lines.purchase_order_id IS 'Purchase order this line belongs to; references purchase_orders.purchase_order_id.';
COMMENT ON COLUMN purchase_order_lines.line_number IS 'Position of the line within its purchase order, starting at 1.';
COMMENT ON COLUMN purchase_order_lines.product_id IS 'Product being purchased; references products.product_id.';
COMMENT ON COLUMN purchase_order_lines.quantity_ordered IS 'Number of units ordered from the supplier.';
COMMENT ON COLUMN purchase_order_lines.quantity_received IS 'Running total of units received so far against this line. Less than quantity_ordered means the line is still open.';
COMMENT ON COLUMN purchase_order_lines.unit_cost IS 'Agreed price per unit paid to the supplier, in the purchase order currency.';
COMMENT ON COLUMN purchase_order_lines.line_total IS 'Computed line amount: quantity_ordered times unit_cost, rounded to 2 decimals.';

CREATE INDEX purchase_order_lines_product_idx ON purchase_order_lines (product_id);

-- -----------------------------------------------------------------------------
-- purchase_receipts
-- -----------------------------------------------------------------------------
CREATE TABLE purchase_receipts (
    receipt_id          bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    po_line_id          bigint NOT NULL REFERENCES purchase_order_lines (po_line_id),
    received_at         timestamptz NOT NULL DEFAULT now(),
    quantity_received   integer NOT NULL CHECK (quantity_received > 0),
    quantity_rejected   integer NOT NULL DEFAULT 0 CHECK (quantity_rejected >= 0),
    received_by         text,
    notes               text
);

COMMENT ON TABLE purchase_receipts IS
'Goods-received records: each time a delivery arrives at the warehouse against a purchase order line. One line can have several receipts if the supplier delivers in parts. Use received_at against purchase_orders.expected_delivery_date to measure on-time delivery.';
COMMENT ON COLUMN purchase_receipts.receipt_id IS 'Surrogate primary key for the receipt.';
COMMENT ON COLUMN purchase_receipts.po_line_id IS 'Purchase order line the delivery is against; references purchase_order_lines.po_line_id.';
COMMENT ON COLUMN purchase_receipts.received_at IS 'Timestamp when the goods physically arrived and were checked in.';
COMMENT ON COLUMN purchase_receipts.quantity_received IS 'Number of units accepted into stock in this delivery.';
COMMENT ON COLUMN purchase_receipts.quantity_rejected IS 'Number of units refused in this delivery because of damage or quality problems. Not added to stock.';
COMMENT ON COLUMN purchase_receipts.received_by IS 'Name or user id of the warehouse employee who received the delivery.';
COMMENT ON COLUMN purchase_receipts.notes IS 'Free-text notes about the delivery, such as damage details.';

CREATE INDEX purchase_receipts_line_idx ON purchase_receipts (po_line_id);

-- -----------------------------------------------------------------------------
-- carriers
-- -----------------------------------------------------------------------------
CREATE TABLE carriers (
    carrier_id              bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    carrier_code            text NOT NULL,
    carrier_name            text NOT NULL,
    tracking_url_template   text,
    is_active               boolean NOT NULL DEFAULT true
);

COMMENT ON TABLE carriers IS
'Shipping companies that transport outbound customer orders, such as parcel and freight carriers. Referenced by shipments.';
COMMENT ON COLUMN carriers.carrier_id IS 'Surrogate primary key for the carrier.';
COMMENT ON COLUMN carriers.carrier_code IS 'Short unique code for the carrier, such as UPS or FEDEX.';
COMMENT ON COLUMN carriers.carrier_name IS 'Full display name of the carrier.';
COMMENT ON COLUMN carriers.tracking_url_template IS 'Web address pattern for tracking a package, with the placeholder {tracking_number} where the shipment tracking number goes.';
COMMENT ON COLUMN carriers.is_active IS 'True if the company currently ships with this carrier.';

-- -----------------------------------------------------------------------------
-- shipments
-- -----------------------------------------------------------------------------
CREATE TABLE shipments (
    shipment_id             bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    shipment_number         text NOT NULL,
    sales_order_id          bigint NOT NULL REFERENCES sales_orders (sales_order_id),
    carrier_id              bigint NOT NULL REFERENCES carriers (carrier_id),
    warehouse_id            bigint NOT NULL REFERENCES warehouses (warehouse_id),
    tracking_number         text,
    shipment_status         text NOT NULL DEFAULT 'label_created'
                            CHECK (shipment_status IN ('label_created', 'in_transit', 'out_for_delivery', 'delivered', 'exception', 'returned')),
    shipped_at              timestamptz,
    estimated_delivery_at   timestamptz,
    delivered_at            timestamptz,
    shipping_cost           numeric(10,2) CHECK (shipping_cost >= 0),
    weight_kg               numeric(10,3) CHECK (weight_kg >= 0),
    created_at              timestamptz NOT NULL DEFAULT now()
);

COMMENT ON TABLE shipments IS
'Outbound packages sent to customers for a sales order, with the carrier, tracking number, and delivery progress. A sales order can have several shipments if it is sent in parts. Compare delivered_at with estimated_delivery_at to measure on-time delivery.';
COMMENT ON COLUMN shipments.shipment_id IS 'Surrogate primary key for the shipment.';
COMMENT ON COLUMN shipments.shipment_number IS 'Unique human-readable shipment number, such as SHP-2026-000789.';
COMMENT ON COLUMN shipments.sales_order_id IS 'Sales order being delivered; references sales_orders.sales_order_id.';
COMMENT ON COLUMN shipments.carrier_id IS 'Carrier transporting the package; references carriers.carrier_id.';
COMMENT ON COLUMN shipments.warehouse_id IS 'Warehouse the package ships from; references warehouses.warehouse_id.';
COMMENT ON COLUMN shipments.tracking_number IS 'Carrier-issued tracking number used to follow the package.';
COMMENT ON COLUMN shipments.shipment_status IS 'Delivery progress: label_created (not yet picked up), in_transit, out_for_delivery, delivered, exception (delay or delivery problem), returned (sent back).';
COMMENT ON COLUMN shipments.shipped_at IS 'Timestamp when the carrier picked up the package. NULL until it ships.';
COMMENT ON COLUMN shipments.estimated_delivery_at IS 'Carrier estimate of when the package will arrive.';
COMMENT ON COLUMN shipments.delivered_at IS 'Timestamp when the package was delivered. NULL until delivered.';
COMMENT ON COLUMN shipments.shipping_cost IS 'Amount charged by the carrier for this shipment, in USD.';
COMMENT ON COLUMN shipments.weight_kg IS 'Total weight of the package in kilograms.';
COMMENT ON COLUMN shipments.created_at IS 'Timestamp when the shipment record was created.';

CREATE INDEX shipments_order_idx ON shipments (sales_order_id);
CREATE INDEX shipments_carrier_idx ON shipments (carrier_id);
CREATE INDEX shipments_status_idx ON shipments (shipment_status);

-- -----------------------------------------------------------------------------
-- shipment_lines
-- -----------------------------------------------------------------------------
CREATE TABLE shipment_lines (
    shipment_line_id    bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    shipment_id         bigint NOT NULL REFERENCES shipments (shipment_id) ON DELETE CASCADE,
    sales_order_line_id bigint NOT NULL REFERENCES sales_order_lines (sales_order_line_id),
    quantity_shipped    integer NOT NULL CHECK (quantity_shipped > 0)
);

COMMENT ON TABLE shipment_lines IS
'Contents of a shipment: which sales order lines, and how many units of each, are in the package. Lets one order line be split across several shipments.';
COMMENT ON COLUMN shipment_lines.shipment_line_id IS 'Surrogate primary key for the shipment line.';
COMMENT ON COLUMN shipment_lines.shipment_id IS 'Shipment the units are packed in; references shipments.shipment_id.';
COMMENT ON COLUMN shipment_lines.sales_order_line_id IS 'Sales order line being fulfilled; references sales_order_lines.sales_order_line_id.';
COMMENT ON COLUMN shipment_lines.quantity_shipped IS 'Number of units of the order line included in this shipment.';

CREATE INDEX shipment_lines_order_line_idx ON shipment_lines (sales_order_line_id);

## Load supply chain demo data

Loads sample data into the `supply_chain` schema created by the previous cell. 

| Area | Tables | Rows |
|---|---|---|
| Reference data | `product_categories`, `suppliers`, `products`, `supplier_products`, `warehouses`, `carriers`, `customers` | ~1,000 |
| Purchasing | `purchase_orders`, `purchase_order_lines`, `purchase_receipts` | 600 / 1,500 / ~1,200 |
| Sales | `sales_orders`, `sales_order_lines` | 5,000 / 15,000 |
| Delivery | `shipments`, `shipment_lines` | ~4,750 / ~14,300 |
| Stock | `inventory`, `inventory_transactions` | 1,020 / ~17,000 |

In [ ]:
%%sql

SELECT setseed(0.42);

-- -----------------------------------------------------------------------------
-- Product categories (4 top level, 12 sub categories)
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.product_categories (category_name, description) VALUES
('Electronics',         'Electronic components, cables, power products, and sensors.'),
('Industrial Supplies', 'Fasteners, hand tools, and safety gear for industrial and facility use.'),
('Packaging',           'Boxes, tape, pallets, and wrap used to pack and ship goods.'),
('Office Products',     'Paper, pens, and desk accessories for offices.');

INSERT INTO supply_chain.product_categories (category_name, parent_category_id, description)
SELECT v.name, p.category_id, v.descr
FROM (VALUES
  ('Cables & Connectors',  'Electronics',         'Data and power cables, plugs, and terminal connectors.'),
  ('Batteries & Power',    'Electronics',         'Batteries, chargers, surge protection, and power distribution.'),
  ('Sensors',              'Electronics',         'Temperature, pressure, motion, and other measurement sensors.'),
  ('Fasteners',            'Industrial Supplies', 'Bolts, screws, washers, rivets, and anchors.'),
  ('Hand Tools',           'Industrial Supplies', 'Manual tools such as wrenches, hammers, and pliers.'),
  ('Safety Equipment',     'Industrial Supplies', 'Personal protective equipment and first aid.'),
  ('Boxes & Cartons',      'Packaging',           'Shipping cartons, mailers, and storage boxes.'),
  ('Tape & Adhesives',     'Packaging',           'Packing tape, masking tape, glue, and adhesives.'),
  ('Pallets & Wrap',       'Packaging',           'Pallets, stretch wrap, strapping, and corner protection.'),
  ('Paper Products',       'Office Products',     'Copy paper, notepads, labels, and envelopes.'),
  ('Writing Instruments',  'Office Products',     'Pens, pencils, markers, and highlighters.'),
  ('Desk Accessories',     'Office Products',     'Staplers, organizers, dispensers, and stands.')
) AS v(name, parent, descr)
JOIN supply_chain.product_categories p ON p.category_name = v.parent;

-- -----------------------------------------------------------------------------
-- Suppliers (25)
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.suppliers (supplier_code, supplier_name, country_code, contact_email, contact_phone,
                                    payment_terms_days, status, rating, created_at, updated_at)
SELECT format('SUP-%s', lpad(g.i::text, 4, '0')),
       (ARRAY['Apex','Summit','Pioneer','Northstar','Bluewater','Ironwood','Keystone','Meridian','Vertex','Harbor'])[1 + (g.i - 1) % 10]
         || ' ' ||
       (ARRAY['Industrial','Components','Supply Co.','Logistics','Manufacturing'])[1 + ((g.i - 1) / 10) % 5],
       (ARRAY['US','US','US','DE','CN','MX','CA','JP','KR','IN','GB','VN'])[1 + floor(g.r_cty * 12)::int],
       format('sales@supplier%s.example.com', g.i),
       format('+1-555-%s', lpad((2000 + g.i)::text, 4, '0')),
       (ARRAY[15, 30, 30, 45, 60, 90])[1 + floor(g.r_pay * 6)::int],
       CASE WHEN g.r_st < 0.84 THEN 'active' WHEN g.r_st < 0.92 THEN 'on_hold' ELSE 'inactive' END,
       round((2.5 + g.r_rate * 2.5)::numeric, 2),
       g.created,
       least(now(), g.created + g.r_upd * interval '300 days')
FROM (
  SELECT i, random() AS r_cty, random() AS r_pay, random() AS r_st, random() AS r_rate, random() AS r_upd,
         now() - (370 + random() * 400) * interval '1 day' AS created
  FROM generate_series(1, 25) AS i
) g
ORDER BY g.i;

-- -----------------------------------------------------------------------------
-- Products (204 = 17 per sub category). The noun lists are inlined below.
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.products (sku, product_name, description, category_id, unit_of_measure,
                                   unit_cost, list_price, weight_kg, is_active, created_at)
SELECT format('SKU-%s', lpad(g.i::text, 5, '0')),
       format('%s %s (Model %s)',
              (ARRAY['Standard','Heavy-Duty','Premium','Compact','Economy','Industrial','Professional','Eco'])[1 + floor(g.r_adj * 8)::int],
              n.nouns[1 + ((g.i - 1) / 12) % array_length(n.nouns, 1)],
              lpad(g.i::text, 3, '0')),
       format('General purpose %s for warehouse, facility, and office use.',
              lower(n.nouns[1 + ((g.i - 1) / 12) % array_length(n.nouns, 1)])),
       n.category_id,
       CASE WHEN g.r_uom < 0.75 THEN 'each' WHEN g.r_uom < 0.90 THEN 'box'
            WHEN g.r_uom < 0.95 THEN 'pallet' WHEN g.r_uom < 0.98 THEN 'kg' ELSE 'liter' END,
       round((2 + g.r_cost * 98)::numeric, 2),
       round(((2 + g.r_cost * 98) * (1.25 + g.r_mult * 0.75))::numeric, 2),
       round((0.05 + g.r_wt * g.r_wt * 20)::numeric, 3),
       g.r_act > 0.05,
       now() - (370 + g.r_cr * 400) * interval '1 day'
FROM (
  SELECT i, random() AS r_adj, random() AS r_uom, random() AS r_cost, random() AS r_mult,
         random() AS r_wt, random() AS r_act, random() AS r_cr
  FROM generate_series(1, 204) AS i
) g
JOIN (
  SELECT row_number() OVER (ORDER BY c.category_id) AS rn, c.category_id, v.nouns
  FROM supply_chain.product_categories c
  JOIN (VALUES
    ('Cables & Connectors', ARRAY['USB-C Cable','HDMI Cable','Ethernet Cable','Power Cord','Terminal Connector','Cable Tie Pack']),
    ('Batteries & Power',   ARRAY['AA Battery Pack','Lithium Battery','Power Strip','Surge Protector','Wall Charger','Extension Cord']),
    ('Sensors',             ARRAY['Temperature Sensor','Proximity Sensor','Pressure Sensor','Motion Detector','Humidity Sensor','Light Sensor']),
    ('Fasteners',           ARRAY['Hex Bolt Set','Machine Screw Kit','Lock Washer Pack','Anchor Bolt','Rivet Assortment','Threaded Rod']),
    ('Hand Tools',          ARRAY['Adjustable Wrench','Claw Hammer','Screwdriver Set','Utility Knife','Pliers','Tape Measure']),
    ('Safety Equipment',    ARRAY['Safety Glasses','Work Gloves','Hard Hat','Ear Protection','High-Visibility Vest','First Aid Kit']),
    ('Boxes & Cartons',     ARRAY['Shipping Carton','Corrugated Box','Mailer Box','Storage Bin','Archive Box','Gift Box']),
    ('Tape & Adhesives',    ARRAY['Packing Tape','Masking Tape','Duct Tape','Glue Stick','Epoxy Adhesive','Double-Sided Tape']),
    ('Pallets & Wrap',      ARRAY['Wooden Pallet','Plastic Pallet','Stretch Wrap Roll','Strapping Band','Corner Protector','Pallet Cover']),
    ('Paper Products',      ARRAY['Copy Paper Ream','Notepad','Sticky Notes','Label Sheets','Envelope Pack','Index Cards']),
    ('Writing Instruments', ARRAY['Ballpoint Pen','Gel Pen','Permanent Marker','Highlighter','Mechanical Pencil','Whiteboard Marker']),
    ('Desk Accessories',    ARRAY['Stapler','Desk Organizer','Paper Clip Box','Tape Dispenser','Monitor Stand','Binder Clips'])
  ) AS v(name, nouns) ON v.name = c.category_name
) n ON n.rn = ((g.i - 1) % 12) + 1
ORDER BY g.i;

-- -----------------------------------------------------------------------------
-- Supplier offers: each product has 1 to 3 suppliers, the first is preferred
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.supplier_products (supplier_id, product_id, supplier_sku, unit_price, currency_code,
                                            lead_time_days, min_order_qty, is_preferred)
SELECT 1 + ((p.product_id * 7 + k.k * 5) % 25),
       p.product_id,
       format('S%s-%s', lpad((1 + ((p.product_id * 7 + k.k * 5) % 25))::text, 2, '0'), lpad(p.product_id::text, 5, '0')),
       round((p.unit_cost * (0.85 + random() * 0.35))::numeric, 2),
       'USD',
       3 + floor(random() * 40)::int,
       (ARRAY[1, 5, 10, 25, 50, 100])[1 + floor(random() * 6)::int],
       k.k = 0
FROM supply_chain.products p
JOIN generate_series(0, 2) AS k(k) ON k.k <= p.product_id % 3;

-- -----------------------------------------------------------------------------
-- Warehouses (5) and carriers (5, the last one inactive)
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.warehouses (warehouse_code, warehouse_name, city, state_province, country_code, capacity_units, is_active) VALUES
('WH-DAL', 'Dallas Distribution Center',  'Dallas',   'TX', 'US', 200000, true),
('WH-ATL', 'Atlanta Fulfillment Center',  'Atlanta',  'GA', 'US', 150000, true),
('WH-REN', 'Reno West Coast Warehouse',   'Reno',     'NV', 'US', 120000, true),
('WH-CMH', 'Columbus Midwest Warehouse',  'Columbus', 'OH', 'US', 100000, true),
('WH-YYZ', 'Toronto Canada Warehouse',    'Toronto',  'ON', 'CA',  80000, true);

INSERT INTO supply_chain.carriers (carrier_code, carrier_name, tracking_url_template, is_active) VALUES
('UPS',   'United Parcel Service',        'https://www.ups.com/track?tracknum={tracking_number}', true),
('FEDEX', 'FedEx',                        'https://www.fedex.com/fedextrack/?trknbr={tracking_number}', true),
('USPS',  'United States Postal Service', 'https://tools.usps.com/go/TrackConfirmAction?tLabels={tracking_number}', true),
('DHL',   'DHL Express',                  'https://www.dhl.com/en/express/tracking.html?AWB={tracking_number}', true),
('REGFR', 'Regional Freight Lines',       NULL, false);

-- -----------------------------------------------------------------------------
-- Customers (300): about half retail, a third wholesale, the rest distributors
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.customers (customer_code, customer_name, customer_type, email, phone,
                                    billing_city, billing_state_province, billing_country_code,
                                    credit_limit, status, created_at)
SELECT format('CUST-%s', lpad(g.i::text, 5, '0')),
       CASE WHEN g.ctype = 'retail'
            THEN (ARRAY['Jordan','Taylor','Morgan','Casey','Riley','Avery','Quinn','Harper','Dakota','Rowan','Sam','Jamie'])[1 + (g.i * 7) % 12]
                 || ' ' ||
                 (ARRAY['Alvarez','Nguyen','Patel','Brooks','Kim','Johnson','Garcia','Murphy','Lopez','Chen','Walker','Hughes'])[1 + (g.i * 5 + g.i / 12) % 12]
            ELSE (ARRAY['Lakeside','Cedar','Redwood','Granite','Beacon','Falcon','Orchard','Crescent','Atlas','Willow'])[1 + (g.i * 3) % 10]
                 || ' ' ||
                 (ARRAY['Trading Co.','Wholesale LLC','Distribution Inc.','Supply Group','Partners LLC','Enterprises'])[1 + (g.i / 4) % 6]
       END,
       g.ctype,
       format('customer%s@example.com', g.i),
       format('+1-555-%s', lpad(g.i::text, 4, '0')),
       geo.city, geo.state, geo.country,
       CASE g.ctype WHEN 'retail'    THEN floor(g.r_cl * 11) * 500
                    WHEN 'wholesale' THEN 25000 + floor(g.r_cl * 16) * 5000
                    ELSE                  100000 + floor(g.r_cl * 9) * 50000 END,
       CASE WHEN g.i > 240 AND g.r_st < 0.15 THEN 'suspended'
            WHEN g.i > 240 AND g.r_st < 0.25 THEN 'closed'
            ELSE 'active' END,
       now() - (366 + g.r_cr * 400) * interval '1 day'
FROM (
  SELECT g0.*, CASE WHEN g0.r_type < 0.50 THEN 'retail' WHEN g0.r_type < 0.85 THEN 'wholesale' ELSE 'distributor' END AS ctype
  FROM (
    SELECT i, random() AS r_type, random() AS r_cl, random() AS r_st, random() AS r_cr,
           1 + floor(random() * 12)::int AS geo_rn
    FROM generate_series(1, 300) AS i
  ) g0
) g
JOIN (VALUES
  (1, 'Dallas', 'TX', 'US'), (2, 'Houston', 'TX', 'US'), (3, 'Atlanta', 'GA', 'US'), (4, 'Chicago', 'IL', 'US'),
  (5, 'Denver', 'CO', 'US'), (6, 'Seattle', 'WA', 'US'), (7, 'Phoenix', 'AZ', 'US'), (8, 'Columbus', 'OH', 'US'),
  (9, 'Miami', 'FL', 'US'), (10, 'Boston', 'MA', 'US'), (11, 'Reno', 'NV', 'US'), (12, 'Toronto', 'ON', 'CA')
) AS geo(rn, city, state, country) ON geo.rn = g.geo_rn
ORDER BY g.i;

-- -----------------------------------------------------------------------------
-- Purchase orders (600). Recent orders go to active suppliers, older ones may
-- go to any supplier.
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.purchase_orders (po_number, supplier_id, warehouse_id, po_status, order_date,
                                          expected_delivery_date, currency_code, created_at, updated_at)
SELECT format('PO-%s-%s', to_char(s.od, 'YYYY'), lpad(s.i::text, 6, '0')),
       s.supplier_id, s.warehouse_id, s.po_status, s.od, s.exp, 'USD',
       s.od + interval '9 hours' + s.r4 * interval '6 hours',
       s.od + interval '9 hours' + s.r4 * interval '6 hours'
FROM (
  SELECT t.*,
         CASE WHEN t.r1 < 0.03 THEN 'cancelled'
              WHEN t.exp > current_date THEN
                   CASE WHEN (current_date - t.od) <= 6 AND t.r2 < 0.5 THEN 'draft'
                        WHEN t.r2 < 0.35 THEN 'submitted'
                        ELSE 'confirmed' END
              WHEN t.r2 < 0.87 THEN 'received'
              WHEN t.r2 < 0.94 THEN 'partially_received'
              ELSE 'confirmed' END AS po_status
  FROM (
    SELECT row_number() OVER (ORDER BY g.od) AS i,
           g.od,
           g.od + (7 + floor(g.r3 * 28))::int AS exp,
           CASE WHEN (current_date - g.od) > 120 AND g.r5 < 0.2
                THEN 1 + floor(g.r6 * (SELECT count(*) FROM supply_chain.suppliers))::int
                ELSE act.supplier_id END AS supplier_id,
           g.warehouse_id, g.r1, g.r2, g.r4
    FROM (
      SELECT current_date - floor(random() * 360)::int AS od,
             random() AS r1, random() AS r2, random() AS r3, random() AS r4,
             random() AS r5, random() AS r6,
             1 + floor(random() * 5)::int AS warehouse_id
      FROM generate_series(1, 600) AS gs(n)
    ) g
    JOIN (
      SELECT supplier_id, row_number() OVER (ORDER BY supplier_id) AS rn
      FROM supply_chain.suppliers WHERE status = 'active'
    ) act ON act.rn = 1 + floor(g.r6 * (SELECT count(*) FROM supply_chain.suppliers WHERE status = 'active'))::int
  ) t
) s
ORDER BY s.i;

-- Lines: 1 to 4 offers drawn from the supplier's own catalog
INSERT INTO supply_chain.purchase_order_lines (purchase_order_id, line_number, product_id, quantity_ordered, unit_cost)
SELECT x.purchase_order_id,
       row_number() OVER (PARTITION BY x.purchase_order_id ORDER BY x.product_id),
       x.product_id,
       x.min_order_qty * (1 + floor(random() * 3))::int,
       x.unit_price
FROM (
  SELECT po.purchase_order_id, sp.product_id, sp.min_order_qty, sp.unit_price,
         row_number() OVER (PARTITION BY po.purchase_order_id ORDER BY random()) AS pick,
         1 + (po.purchase_order_id % 4) AS n_lines
  FROM supply_chain.purchase_orders po
  JOIN supply_chain.supplier_products sp ON sp.supplier_id = po.supplier_id
) x
WHERE x.pick <= x.n_lines;

-- Goods receipts. Delivery date = promised date plus noise plus a lateness
-- penalty that grows as the supplier rating falls. Partial POs get half of line 1.
INSERT INTO supply_chain.purchase_receipts (po_line_id, received_at, quantity_received, quantity_rejected, received_by, notes)
SELECT pol.po_line_id,
       least(po.expected_delivery_date + interval '10 hours'
               + ((((po.purchase_order_id * 7919) % 100) / 100.0 * 6 - 2) + (5 - s.rating)::float8 * 2.5) * interval '1 day',
             now() - interval '1 hour'),
       CASE WHEN po.po_status = 'received' THEN pol.quantity_ordered
            ELSE greatest(1, pol.quantity_ordered / 2) END,
       CASE WHEN pol.rr < 0.08 THEN 1 + floor(pol.rr2 * pol.quantity_ordered * 0.1)::int ELSE 0 END,
       (ARRAY['mlopez','jchen','sparks','dnguyen','tbrooks'])[1 + floor(pol.rr3 * 5)::int],
       CASE WHEN pol.rr < 0.08 THEN 'Some units rejected: damaged in transit.' END
FROM (
  SELECT l.po_line_id, l.purchase_order_id, l.line_number, l.quantity_ordered,
         random() AS rr, random() AS rr2, random() AS rr3
  FROM supply_chain.purchase_order_lines l
) pol
JOIN supply_chain.purchase_orders po ON po.purchase_order_id = pol.purchase_order_id
JOIN supply_chain.suppliers s ON s.supplier_id = po.supplier_id
WHERE po.po_status = 'received'
   OR (po.po_status = 'partially_received' AND pol.line_number = 1);

UPDATE supply_chain.purchase_order_lines pol
SET quantity_received = x.q
FROM (SELECT po_line_id, sum(quantity_received) AS q
      FROM supply_chain.purchase_receipts GROUP BY po_line_id) x
WHERE x.po_line_id = pol.po_line_id;

UPDATE supply_chain.purchase_orders po
SET total_amount = x.t
FROM (SELECT purchase_order_id, sum(line_total) AS t
      FROM supply_chain.purchase_order_lines GROUP BY purchase_order_id) x
WHERE x.purchase_order_id = po.purchase_order_id;

UPDATE supply_chain.purchase_orders po
SET updated_at = x.last_rcpt
FROM (SELECT pol.purchase_order_id, max(pr.received_at) AS last_rcpt
      FROM supply_chain.purchase_receipts pr
      JOIN supply_chain.purchase_order_lines pol ON pol.po_line_id = pr.po_line_id
      GROUP BY pol.purchase_order_id) x
WHERE x.purchase_order_id = po.purchase_order_id;

-- -----------------------------------------------------------------------------
-- Sales orders (5,000 over the last 364 days). Status follows the order age.
-- Customer ids are skewed so a few customers place most orders. About 80
-- percent of orders ship to the customer billing address.
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.sales_orders (order_number, customer_id, warehouse_id, order_status, order_date,
                                       requested_delivery_date, ship_to_city, ship_to_state_province,
                                       ship_to_country_code, currency_code, created_at, updated_at)
SELECT format('SO-%s-%s', to_char(s.od, 'YYYY'), lpad(s.i::text, 6, '0')),
       s.customer_id, s.warehouse_id, s.status, s.od,
       s.od::date + 5 + floor(s.r3 * 10)::int,
       CASE WHEN s.ship_to_billing THEN c.billing_city ELSE geo.city END,
       CASE WHEN s.ship_to_billing THEN c.billing_state_province ELSE geo.state END,
       CASE WHEN s.ship_to_billing THEN c.billing_country_code ELSE geo.country END,
       'USD', s.od,
       CASE WHEN s.status = 'pending' THEN s.od
            ELSE least(s.od + s.r4 * interval '3 days', now()) END
FROM (
  SELECT t.*,
         CASE WHEN t.r1 < 0.04 THEN 'cancelled'
              WHEN t.age_days > 10 THEN 'delivered'
              WHEN t.age_days > 6  THEN CASE WHEN t.r2 < 0.8 THEN 'delivered' ELSE 'shipped' END
              WHEN t.age_days > 3  THEN CASE WHEN t.r2 < 0.6 THEN 'shipped' WHEN t.r2 < 0.8 THEN 'delivered' ELSE 'picking' END
              WHEN t.age_days > 1.5 THEN CASE WHEN t.r2 < 0.35 THEN 'shipped' WHEN t.r2 < 0.7 THEN 'picking' ELSE 'confirmed' END
              WHEN t.age_days > 0.5 THEN CASE WHEN t.r2 < 0.5 THEN 'confirmed' ELSE 'pending' END
              ELSE 'pending' END AS status
  FROM (
    SELECT row_number() OVER (ORDER BY g.od) AS i, g.od,
           extract(epoch FROM (now() - g.od)) / 86400.0 AS age_days,
           g.customer_id, g.warehouse_id,
           g.r5 < 0.8 AS ship_to_billing,
           1 + floor(g.r7 * 12)::int AS ship_geo_rn,
           g.r1, g.r2, g.r3, g.r4
    FROM (
      SELECT now() - random() * interval '364 days' AS od,
             1 + floor(power(random(), 1.6) * 300)::int AS customer_id,
             1 + floor(random() * 5)::int AS warehouse_id,
             random() AS r1, random() AS r2, random() AS r3, random() AS r4,
             random() AS r5, random() AS r7
      FROM generate_series(1, 5000) AS gs(n)
    ) g
  ) t
) s
JOIN supply_chain.customers c ON c.customer_id = s.customer_id
JOIN (VALUES
  (1, 'Dallas', 'TX', 'US'), (2, 'Houston', 'TX', 'US'), (3, 'Atlanta', 'GA', 'US'), (4, 'Chicago', 'IL', 'US'),
  (5, 'Denver', 'CO', 'US'), (6, 'Seattle', 'WA', 'US'), (7, 'Phoenix', 'AZ', 'US'), (8, 'Columbus', 'OH', 'US'),
  (9, 'Miami', 'FL', 'US'), (10, 'Boston', 'MA', 'US'), (11, 'Reno', 'NV', 'US'), (12, 'Toronto', 'ON', 'CA')
) AS geo(rn, city, state, country) ON geo.rn = s.ship_geo_rn
ORDER BY s.i;

-- Lines: 1 to 5 distinct active products, with some products far more popular
INSERT INTO supply_chain.sales_order_lines (sales_order_id, line_number, product_id, quantity_ordered, unit_price, discount_pct)
SELECT x.sales_order_id,
       row_number() OVER (PARTITION BY x.sales_order_id ORDER BY x.product_id),
       x.product_id,
       1 + floor(power(random(), 2) * 20)::int,
       x.list_price,
       (ARRAY[0, 0, 0, 0, 5, 5, 10, 15])[1 + floor(random() * 8)::int]
FROM (
  SELECT so.sales_order_id, p.product_id, p.list_price,
         row_number() OVER (PARTITION BY so.sales_order_id
                            ORDER BY random() * (1.0 + (p.product_id % 10) / 3.0)) AS pick,
         1 + (so.sales_order_id % 5) AS n_lines
  FROM supply_chain.sales_orders so
  JOIN supply_chain.products p ON p.is_active
) x
WHERE x.pick <= x.n_lines;

UPDATE supply_chain.sales_orders so
SET total_amount = x.t
FROM (SELECT sales_order_id, sum(line_total) AS t
      FROM supply_chain.sales_order_lines GROUP BY sales_order_id) x
WHERE x.sales_order_id = so.sales_order_id;

-- -----------------------------------------------------------------------------
-- Shipments for shipped and delivered orders. About one in ten multi-line
-- orders is split into two shipments. Slower carriers get a lateness bias.
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.shipments (shipment_number, sales_order_id, carrier_id, warehouse_id, tracking_number,
                                    shipment_status, shipped_at, estimated_delivery_at, delivered_at, created_at)
SELECT format('SHP-%s-%s', lpad(t.sales_order_id::text, 6, '0'), t.seq),
       t.sales_order_id, t.carrier_id, t.warehouse_id,
       c.carrier_code || lpad((t.sales_order_id * 7919 + t.seq)::text, 10, '0'),
       CASE WHEN t.order_status = 'delivered' THEN 'delivered'
            WHEN t.r4 < 0.45 THEN 'in_transit'
            WHEN t.r4 < 0.75 THEN 'out_for_delivery'
            WHEN t.r4 < 0.90 THEN 'exception'
            ELSE 'in_transit' END,
       t.shipped_at,
       t.shipped_at + t.est_days * interval '1 day',
       CASE WHEN t.order_status = 'delivered'
            THEN least(t.shipped_at + t.transit_days * interval '1 day', now() - interval '30 minutes') END,
       t.shipped_at - interval '4 hours'
FROM (
  SELECT p.sales_order_id, p.warehouse_id, p.order_status, p.seq, p.carrier_id, p.r4,
         least(p.order_date + interval '6 hours' + p.r1 * interval '30 hours'
                 + (p.seq - 1) * interval '1 day',
               now() - interval '1 hour') AS shipped_at,
         (2 + floor(p.r2 * 4))::int AS est_days,
         greatest(0.5, (2 + floor(p.r2 * 4)) + (p.r3 * 2 - 1.7)
                       + CASE p.carrier_id WHEN 3 THEN 0.9 WHEN 4 THEN 0.4 WHEN 2 THEN 0.15 ELSE 0 END) AS transit_days
  FROM (
    SELECT so.sales_order_id, so.warehouse_id, so.order_status, so.order_date, sq.seq,
           1 + floor(power(random(), 1.4) * 4)::int AS carrier_id,
           random() AS r1, random() AS r2, random() AS r3, random() AS r4
    FROM supply_chain.sales_orders so
    LEFT JOIN (SELECT sales_order_id, count(*) AS n
               FROM supply_chain.sales_order_lines GROUP BY sales_order_id) lc
           ON lc.sales_order_id = so.sales_order_id
    JOIN generate_series(1, 2) AS sq(seq)
           ON sq.seq <= CASE WHEN so.sales_order_id % 10 = 0 AND lc.n >= 2 THEN 2 ELSE 1 END
    WHERE so.order_status IN ('shipped', 'delivered')
  ) p
) t
JOIN supply_chain.carriers c ON c.carrier_id = t.carrier_id
ORDER BY t.sales_order_id, t.seq;

INSERT INTO supply_chain.shipment_lines (shipment_id, sales_order_line_id, quantity_shipped)
SELECT s.shipment_id, sol.sales_order_line_id, sol.quantity_ordered
FROM supply_chain.shipments s
JOIN supply_chain.sales_order_lines sol ON sol.sales_order_id = s.sales_order_id
JOIN (SELECT sales_order_id, count(*) AS n
      FROM supply_chain.shipments GROUP BY sales_order_id) sc
  ON sc.sales_order_id = s.sales_order_id
WHERE sc.n = 1
   OR (right(s.shipment_number, 1) = '1' AND sol.line_number % 2 = 1)
   OR (right(s.shipment_number, 1) = '2' AND sol.line_number % 2 = 0);

UPDATE supply_chain.shipments s
SET weight_kg = x.w,
    shipping_cost = round((4 + x.w * 0.9 + random() * 8)::numeric, 2)
FROM (SELECT sl.shipment_id,
             round(sum(sl.quantity_shipped * coalesce(p.weight_kg, 0.5))::numeric, 3) AS w
      FROM supply_chain.shipment_lines sl
      JOIN supply_chain.sales_order_lines sol ON sol.sales_order_line_id = sl.sales_order_line_id
      JOIN supply_chain.products p ON p.product_id = sol.product_id
      GROUP BY sl.shipment_id) x
WHERE x.shipment_id = s.shipment_id;

-- Order last-modified time follows its shipments
UPDATE supply_chain.sales_orders so
SET updated_at = greatest(so.updated_at, x.last_event)
FROM (SELECT sales_order_id, max(coalesce(delivered_at, shipped_at)) AS last_event
      FROM supply_chain.shipments GROUP BY sales_order_id) x
WHERE x.sales_order_id = so.sales_order_id;

-- -----------------------------------------------------------------------------
-- Stock ledger: receipts, shipments, returns, transfers, adjustments, then an
-- opening balance sized so stock never goes negative and reserved stock fits.
-- -----------------------------------------------------------------------------
INSERT INTO supply_chain.inventory_transactions (warehouse_id, product_id, transaction_type, quantity_change,
                                                 reference_type, reference_id, notes, created_at)
SELECT po.warehouse_id, pol.product_id, 'receipt', pr.quantity_received,
       'purchase_order', po.purchase_order_id, NULL, pr.received_at
FROM supply_chain.purchase_receipts pr
JOIN supply_chain.purchase_order_lines pol ON pol.po_line_id = pr.po_line_id
JOIN supply_chain.purchase_orders po ON po.purchase_order_id = pol.purchase_order_id;

INSERT INTO supply_chain.inventory_transactions (warehouse_id, product_id, transaction_type, quantity_change,
                                                 reference_type, reference_id, notes, created_at)
SELECT s.warehouse_id, sol.product_id, 'shipment', -sl.quantity_shipped,
       'sales_order', s.sales_order_id, NULL, s.shipped_at
FROM supply_chain.shipment_lines sl
JOIN supply_chain.shipments s ON s.shipment_id = sl.shipment_id
JOIN supply_chain.sales_order_lines sol ON sol.sales_order_line_id = sl.sales_order_line_id;

INSERT INTO supply_chain.inventory_transactions (warehouse_id, product_id, transaction_type, quantity_change,
                                                 reference_type, reference_id, notes, created_at)
SELECT s.warehouse_id, sol.product_id, 'return', greatest(1, sl.quantity_shipped / 2),
       'sales_order', s.sales_order_id, 'Customer return',
       least(s.delivered_at + interval '3 days', now())
FROM supply_chain.shipment_lines sl
JOIN supply_chain.shipments s ON s.shipment_id = sl.shipment_id
JOIN supply_chain.sales_order_lines sol ON sol.sales_order_line_id = sl.sales_order_line_id
WHERE s.shipment_status = 'delivered' AND random() < 0.015;

-- Warehouse-to-warehouse transfers (60 pairs)
INSERT INTO supply_chain.inventory_transactions (warehouse_id, product_id, transaction_type, quantity_change,
                                                 reference_type, reference_id, notes, created_at)
SELECT CASE d.dir WHEN 'out' THEN d.from_wh ELSE d.to_wh END,
       d.product_id,
       CASE d.dir WHEN 'out' THEN 'transfer_out' ELSE 'transfer_in' END,
       CASE d.dir WHEN 'out' THEN -d.qty ELSE d.qty END,
       'manual', NULL::bigint,
       CASE d.dir WHEN 'out' THEN 'Transfer to ' || wt.warehouse_code ELSE 'Transfer from ' || wf.warehouse_code END,
       d.created_at
FROM (
  SELECT g.i,
         1 + (g.i % 5) AS from_wh,
         1 + (((g.i % 5) + 1 + (g.i % 4)) % 5) AS to_wh,
         1 + (g.i * 13) % (SELECT count(*) FROM supply_chain.products) AS product_id,
         10 + (g.i * 7) % 60 AS qty,
         now() - ((g.i * 5) % 300 + 1) * interval '1 day' AS created_at,
         dir.dir
  FROM generate_series(1, 60) AS g(i)
  CROSS JOIN (VALUES ('out'), ('in')) AS dir(dir)
) d
JOIN supply_chain.warehouses wf ON wf.warehouse_id = d.from_wh
JOIN supply_chain.warehouses wt ON wt.warehouse_id = d.to_wh;

-- Cycle count adjustments (120)
INSERT INTO supply_chain.inventory_transactions (warehouse_id, product_id, transaction_type, quantity_change,
                                                 reference_type, reference_id, notes, created_at)
SELECT 1 + (g.i % 5),
       1 + (g.i * 17) % (SELECT count(*) FROM supply_chain.products),
       'adjustment',
       CASE WHEN g.i % 3 = 0 THEN 1 + (g.i % 8) ELSE -(1 + (g.i % 6)) END,
       'manual', NULL::bigint,
       CASE WHEN g.i % 3 = 0 THEN 'Cycle count correction: found extra units.'
            ELSE 'Cycle count correction: damaged or missing units.' END,
       now() - ((g.i * 3) % 330 + 1) * interval '1 day'
FROM generate_series(1, 120) AS g(i);

-- Opening balance for every warehouse and product. It covers outflows, units
-- reserved for open orders (confirmed or picking), and a random buffer, so a
-- few rows end up near or below their reorder point.
INSERT INTO supply_chain.inventory_transactions (warehouse_id, product_id, transaction_type, quantity_change,
                                                 reference_type, reference_id, notes, created_at)
SELECT w.warehouse_id, p.product_id, 'adjustment',
       greatest(0, -coalesce(net.n, 0)) + coalesce(res.reserved, 0)
         + 5 + floor(power(random(), 0.6) * 400)::int,
       'manual', NULL::bigint, 'Opening balance', now() - interval '400 days'
FROM supply_chain.warehouses w
CROSS JOIN supply_chain.products p
LEFT JOIN (SELECT warehouse_id, product_id, sum(quantity_change) AS n
           FROM supply_chain.inventory_transactions GROUP BY warehouse_id, product_id) net
       ON net.warehouse_id = w.warehouse_id AND net.product_id = p.product_id
LEFT JOIN (SELECT so.warehouse_id, sol.product_id, sum(sol.quantity_ordered)::int AS reserved
           FROM supply_chain.sales_orders so
           JOIN supply_chain.sales_order_lines sol ON sol.sales_order_id = so.sales_order_id
           WHERE so.order_status IN ('confirmed', 'picking')
           GROUP BY so.warehouse_id, sol.product_id) res
       ON res.warehouse_id = w.warehouse_id AND res.product_id = p.product_id;

-- Current stock = sum of the ledger
INSERT INTO supply_chain.inventory (warehouse_id, product_id, quantity_on_hand, quantity_reserved,
                                    reorder_point, reorder_quantity, last_counted_at, updated_at)
SELECT t.warehouse_id, t.product_id, t.on_hand,
       least(coalesce(res.reserved, 0), t.on_hand),
       40 + floor(random() * 60)::int,
       100 + floor(random() * 4)::int * 50,
       now() - random() * interval '90 days',
       t.last_txn
FROM (SELECT warehouse_id, product_id, sum(quantity_change)::int AS on_hand, max(created_at) AS last_txn
      FROM supply_chain.inventory_transactions GROUP BY warehouse_id, product_id) t
LEFT JOIN (SELECT so.warehouse_id, sol.product_id, sum(sol.quantity_ordered)::int AS reserved
           FROM supply_chain.sales_orders so
           JOIN supply_chain.sales_order_lines sol ON sol.sales_order_id = so.sales_order_id
           WHERE so.order_status IN ('confirmed', 'picking')
           GROUP BY so.warehouse_id, sol.product_id) res
       ON res.warehouse_id = t.warehouse_id AND res.product_id = t.product_id;


## Create the Model

This uses Claude Sonnet 5. The model is used for the reasoning when you ask questions via an agent: deciding which tools to call, writing the SQL, and explaining the result. The embedding model used by the KB is separate and is configured in the previous step.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_budget_semantic_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Create Semantic KB

`aidb.create_semantic_kb` crawls the listed schemas and builds the searchable index.

- `name`: the identifier agents use to refer to this KB (`demo_agent_budget_semantic_kb`).
- `model`: the embedding model (`bge-small-en-v1.5-f16`) used to turn table, column, and comment text into vectors.
- `schemas`: which schemas to index. Here, only `supply_chain`.
- `auto_processing => 'Live'`: keeps the KB current. DDL triggers re-crawl affected relations as soon as the schema changes, so adding a column or editing a comment is reflected without any manual step.

The first statement removes an existing KB with the same name so the cell can be re-run. You can list the KBs in your database at any time with `aidb.list_semantic_kbs()`.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'budget_semantic_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.list_semantic_kbs() WHERE name = v_name) THEN
        PERFORM aidb.delete_semantic_kb(v_name);
    END IF;

    PERFORM aidb.create_semantic_kb(
        name    => v_name,
        model   => 'bge-small-en-v1.5-f16',
        schemas => ARRAY['supply_chain'],
        auto_processing => 'Live'
    );
END
$do$;



## Create the Agent

Using the Claude model just created above, create an agent.

Notice that **five tools** are used here. Together they let the agent find the right tables, work out how they connect, and then query them:

| Tool | Role |
|---|---|
| `semantic_kb_search` | Find relevant tables and columns by meaning |
| `search_by_comment` | Search using the descriptions in object comments |
| `list_relationships` | See how tables are linked by foreign keys |
| `find_join_path` | Work out the route for joining tables that are not directly related |
| `run_sql_query` | Run the read-only query the agent writes |

The `instructions` spell out a process for the agent to follow: search the KB, find a join path if needed, write one read-only `SELECT` using only objects it found, run it, and answer briefly while showing the SQL. They also include the key guardrail, *never guess table or column names*. `max_iterations => 15` caps how many tool-calling steps the agent can take for a single question.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_budget_semantic_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;

    PERFORM aidb.create_agent(
      name         => v_name,
      instructions => $$You answer questions about a supply chain schema.
    All data lives in the "supply_chain" schema. Ignore all other schemas.
    The knowledge base is always kb_name => 'budget_semantic_kb'. Pass it on every
    semantic_kb_search, search_by_comment, list_relationships and find_join_path call.
    
    Process:
    1. Call semantic_kb_search to find the relevant tables and columns. Reuse names you
       already found earlier in this conversation instead of searching again.
    2. If the tables are not directly related, call find_join_path to get the join route.
    3. Write one read-only SELECT using only objects you found. Always schema-qualify
       tables, for example supply_chain.products.
    4. Run it with run_sql_query. If it fails on an unknown column, query
       information_schema.columns for that table. Do not guess.
    5. Answer briefly and show the SQL you ran.
    
    Never guess table or column names. If the schema does not contain what is needed, say so.$$,
      model          => 'claude_model_budget_semantic_kb',
      tools          => ARRAY['semantic_kb_search', 'search_by_comment', 'list_relationships',
                              'find_join_path', 'run_sql_query'],
      max_iterations => 15
    );
END
$do$;


## Ask the Agent a Question


**Patterns built in for the demo agent to find**
- Lower-rated suppliers deliver later than promised (about 6 days late near a 2.9 rating, under 2 days near 4.9).
- Carrier on-time delivery varies: UPS about 86%, FedEx about 78%, DHL about 64%, USPS about 39%.
- About 3% of inventory rows are at or below their reorder point.
- About 85 purchase orders are overdue (confirmed or partially received and past the expected date).
- A few customers place most of the orders.
- A handful of shipments are in `exception` status.
    

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "demo_agent_budget_semantic_kb"

_, conversation_id = ask(AGENT, "What are the tables available with this agent?")

In [ ]:
_, conversation_id = ask(AGENT, "How many purchase orders are overdue?")

## Follow-up question using the conversation_id

Now that we have an answer, ask a follow-up. Passing the `conversation_id` gives the agent the earlier messages as context.

Notice how the LLM has to figure out what I mean by "largest one"? The question never says current *what*. The agent has to resolve it from the conversation and from the schema.

In [ ]:
_, conversation_id = ask(AGENT, "Which is the largest one?", conversation_id)

## Conversation Tokens

In [ ]:
%%sql
SELECT task_id, min(created_at) AS task_start,
       count(*) FILTER (WHERE action_type = 'model_request') AS iterations,
       count(*) FILTER (WHERE action_type = 'compaction')    AS compactions,
       coalesce(sum(length(request_payload::text))
                FILTER (WHERE action_type IN ('model_request', 'compaction')), 0) / 4 AS est_input_tokens,
       coalesce(sum(length(response_payload::text))
                FILTER (WHERE action_type IN ('model_response', 'compaction')), 0) / 4
     + coalesce(sum(length(request_payload::text))
                FILTER (WHERE action_type = 'tool_call'), 0) / 4 AS est_output_tokens
FROM aidb_internal.action_log
WHERE conversation_id = :conversation_id
GROUP BY task_id
ORDER BY task_start;


## Update the Agent
Set some budget levels for the Agent to see how it changes the behavior.

In [ ]:
%%sql
SELECT aidb.update_agent('demo_agent_budget_semantic_kb',
    max_iterations => 2,
    budget_strategy => 'error');

## Ask a new question

This should exceed the number of iterations and cause an error.

In [ ]:
_, conversation_id = ask(AGENT, "How many items are in each warehouse?")

## Update the Agent
Change the budget strategy to `summarize`.

In [ ]:
%%sql
SELECT aidb.update_agent('demo_agent_budget_semantic_kb',
    max_iterations => 2,
    budget_strategy => 'summarize');

In [ ]:
_, conversation_id = ask(AGENT, "How many items are in each warehouse?")

## More controls
In addition to setting the `max_iterations`, you can also set:
* `input_token_budget`
* `output_token_budget` 
